# Sprint 3 — Fine-tune the head on YOUR data (keep Gustking backbone)

Strategy: **do NOT swap the model**. Fine-tune the head on our own volunteer+clone data (one TTS engine). **Leave-one-speaker-out** validation guarantees the held-out speaker's real+cloned clips are never in training — so the numbers are about real-vs-synthetic generalization, not speaker ID leakage.

Compute budget: LOSO folds train **only the head** (fast); the single final deployable model uses the fuller recipe. Run cells 1→2→3→4.

In [ ]:
# @title 1. Mount Drive + clone repo + hydrate dataset (same as sprint0/1/2)
from google.colab import drive
from pathlib import Path
import sys, os, pathlib, subprocess, random, shutil

drive.mount("/content/drive")

REPO_URL = "https://github.com/io-PEAK/VoxDetect.git"
REPO_DIR = Path("/content/VoxDetect")

ML_BASE        = Path("/content/drive/MyDrive/VoxDetect/ml-core")
DATASET_DIR    = ML_BASE / "dataset"
RESULTS_DIR    = ML_BASE / "results"
CHECKPOINT_DIR = ML_BASE / "checkpoints"
results_dir = RESULTS_DIR
for d in (DATASET_DIR, RESULTS_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_DIR = Path("/content/VoxDetect_data")

if not (REPO_DIR / "ml-core" / "src").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

SRC_PKG = REPO_DIR / "ml-core" / "src"
sys.path.insert(0, str(SRC_PKG))

subprocess.run([
    sys.executable, str(REPO_DIR / "ml-core" / "scripts" / "organize_dataset.py"),
    "--raw-dir", str(DATASET_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
])
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
test_data_dir = LOCAL_DATA_DIR

!pip install -q torch torchaudio librosa soundfile transformers resemblyzer huggingface_hub numpy scipy
print("deps installed")

In [ ]:
# @title 2. Verify the folder depth for the speaker-partition guarantee
# SPEAKER must be the folder DIRECTLY above each clip (real/<lang>/<speaker>/clip.wav).
# If it isn't, finetune_head.py will hard-fail loudly (it asserts the exact depth)
# rather than silently sorting clips by the wrong folder.
import glob
real_clips = sorted(glob.glob(str(test_data_dir / "real/**/*.wav"), recursive=True))
cloned_clips = sorted(glob.glob(str(test_data_dir / "cloned/**/*.wav"), recursive=True))
print("real clips:", len(real_clips), " cloned clips:", len(cloned_clips))

# Show the actual relative path (indent) so we can SEE which level is the speaker.
print("\nSample real paths (one level deeper == different folder layout):")
for p in real_clips[:6]:
    print("   ", pathlib.Path(p).relative_to(test_data_dir))
print("\nSample cloned paths:")
for p in cloned_clips[:6]:
    print("   ", pathlib.Path(p).relative_to(test_data_dir))

# Detect speaker names (the folder DIRECTLY above each clip) and show counts.
def speakers_of(label):
    d = {}
    for x in glob.glob(str(test_data_dir / f"{label}/**/*.wav"), recursive=True):
        parent = str(pathlib.Path(x).parent)          # glob gives str -> wrap in Path
        d[parent] = d.get(parent, 0) + 1
    return d
real_sp, cloned_sp = speakers_of("real"), speakers_of("cloned")
print("\nSpeaker folders found (must be the clip's parent dir):")
for root, cnt in sorted(real_sp.items(), key=lambda kv: -kv[1]):
    rel = pathlib.Path(root).relative_to(test_data_dir)
    print(f"   real   {rel}  ({cnt} clips)")


# Pick one holdout speaker for the A/B comparison (the LAST real speaker by name).
import pathlib as _p
holdout = _p.Path(max(real_sp, key=lambda k: _p.Path(k).name)).name
print("\n[holdout] using speaker:", holdout)

# Build a real/ + cloned/ dir containing ONLY that speaker, for evaluate.py A/B.
def build_holdout_dir(sp):
    out = pathlib.Path("/content/VoxDetect_holdout")
    if out.exists(): shutil.rmtree(out)
    for i, label in enumerate(("real", "cloned")):
        idx = 0
        for p in glob.glob(str(test_data_dir / f"{label}/**/*.wav"), recursive=True):
            p = pathlib.Path(p)
            if p.parent.name == sp:          # speaker is the clip's parent dir
                lang = p.parent.parent.name  # real/<lang>/<speaker>/
                dest = out / label / f"{lang}_{idx:03d}.wav"
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, dest)
                idx += 1
    return out
holdout_dir = build_holdout_dir(holdout)
print("holdout eval dir:", holdout_dir, "(real+cloned, speaker only)")

In [ ]:
# @title CALIBRATE — pick ONE robust cutoff across ALL speakers (fast: model copied to /content)
import json, pathlib, subprocess, shutil, glob

ML_BASE = pathlib.Path("/content/drive/MyDrive/VoxDetect/ml-core")
CHECKPOINT_DIR = ML_BASE / "checkpoints"
SRC_PKG = pathlib.Path("/content/VoxDetect/ml-core/src")
LOCAL_DATA_DIR = pathlib.Path("/content/VoxDetect_data")
FINAL_MODEL_DIR = CHECKPOINT_DIR / "ft_head_v1"
LOCAL_MODEL_DIR = pathlib.Path("/content/ft_head_v1_local")
assert FINAL_MODEL_DIR.exists(), f"final model dir not found at {FINAL_MODEL_DIR} — run cell 3 then cell 4."

print("Checking the 3 loadable files on Drive:")
for f in ("config.json", "model.safetensors", "preprocessor_config.json"):
    p = FINAL_MODEL_DIR / f
    size = p.stat().st_size if p.exists() else 0
    flag = "OK" if (p.exists() and size > 0) else ("MISSING" if not p.exists() else "ZERO-BYTES")
    print(f"  {f:28s} {size:>12,} bytes  [{flag}]")
    assert flag == "OK", f"{FINAL_MODEL_DIR} incomplete ({f} {flag}). Re-run cell 4 (or cell 3)."

if LOCAL_MODEL_DIR.exists():
    shutil.rmtree(LOCAL_MODEL_DIR)
print("\nCopying model to local disk (~" + str((FINAL_MODEL_DIR/"model.safetensors").stat().st_size//1000000) + " MB)...")
shutil.copytree(str(FINAL_MODEL_DIR), str(LOCAL_MODEL_DIR))
print("local model ready:", LOCAL_MODEL_DIR)

speakers = {}
for x in glob.glob(str(LOCAL_DATA_DIR / "real/**/*.wav"), recursive=True):
    p = pathlib.Path(x)
    speakers.setdefault(p.parent.name, 0)
    speakers[p.parent.name] += 1
speakers = dict(sorted(speakers.items()))
print("\nSpeakers:", speakers)

def find_threshold(sp):
    hol = pathlib.Path("/content/VoxDetect_holdout")
    if hol.exists():
        shutil.rmtree(hol)
    for label in ("real", "cloned"):
        idx = 0
        for p in glob.glob(str(LOCAL_DATA_DIR / f"{label}/**/*.wav"), recursive=True):
            p = pathlib.Path(p)
            if p.parent.name == sp:
                d = hol / label / f"clip_{idx:03d}.wav"
                d.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, d)
                idx += 1
    print(f"  per-clip scores:")
    subprocess.run(["python3", "-m", "evaluate", "--root", str(hol),
                    "--checkpoint", str(LOCAL_MODEL_DIR), "--scores"], cwd=str(SRC_PKG))
    r = subprocess.run(["python3", "-m", "evaluate", "--root", str(hol),
                        "--checkpoint", str(LOCAL_MODEL_DIR), "--find-threshold", "--json"],
                       cwd=str(SRC_PKG), capture_output=True, text=True)

    try:
        if r.returncode != 0:
            print(f"  [FAIL] {sp}: evaluate command failed with return code {r.returncode}")
            print(f"  STDOUT: {r.stdout.strip()}")
            print(f"  STDERR: {r.stderr.strip()}")
            return None
        elif not r.stdout.strip():
            print(f"  [FAIL] {sp}: evaluate command produced empty stdout.")
            print(f"  STDERR: {r.stderr.strip()}")
            return None
        else:
            idx = r.stdout.find('{')
            json_text = r.stdout[idx:] if idx != -1 else r.stdout
            return json.loads(json_text)
    except json.JSONDecodeError as e:
        print(f"  [FAIL] {sp}: JSONDecodeError: {e}")
        print(f"  STDOUT (start): {r.stdout.strip()[:500]}")
        print(f"  STDERR: {r.stderr.strip()}")
        return None

print("=" * 62)
print("Per-speaker best cutoff (max acc, tie-break lower FPR)")
print("=" * 62)
results = []
for sp in speakers:
    print("\n--- " + sp + " ---")
    pay = find_threshold(sp)
    if pay is None:
        continue
    thr = pay["best"]["thr"]
    acc = pay["best"]["acc"]
    fpr = pay["best"]["fpr"]
    n = pay["n_clips"]
    results.append({"speaker": sp, "thr": thr, "acc": acc, "fpr": fpr, "n": n})
    print(f"  cutoff={thr}  ACC={acc*100:.0f}%  FPR={fpr*100:.0f}%  (n={n})")

print()
print("=" * 62)
if results:
    thr_list = sorted(x["thr"] for x in results)
    med = thr_list[len(thr_list)//2]
    print("Per-speaker cutoffs:", thr_list)
    print(f"  median = {med}   min = {thr_list[0]}   max = {thr_list[-1]}")
    print(f"\n✅ SAFE DEMO THRESH = {med}   (median of all per-speaker cutoffs)")
    print("Set THRESH in live_demo.ipynb cell 2 to this value (currently 34.0).")
    print("If any speaker's clones miss at the median, nudge down toward min.")
else:
    print("No speaker thresholds computed — check STDOUT/STDERR above.")
print("=" * 62)


In [ ]:
# @title 3. LOSO cross-validation + final fine-tuned model (dual freeze schedules)
# LOSO folds train ONLY the head (fast, generalization signal). The FINAL model (the one
# that ships) is trained on all non-holdout speakers with the fuller recipe.
!python3 {REPO_DIR}/ml-core/scripts/finetune_head.py \
    --data-dir {test_data_dir} \
    --out-dir {CHECKPOINT_DIR}/ft_head_v1 \
    --epochs 5 --batch-size 4 --lr 1e-5 --augment \
    --holdout {holdout} \
    --results-csv {results_dir}/ablation_results.csv
print("\nDone. LOSO aggregates + holdout eval saved to", CHECKPOINT_DIR / "ft_head_v1" / "finetune_report.json")

In [ ]:
# @title 4. Cleanup — promote best model to root + purge crash-recovery checkpoints
# The live_demo loads the NEWEST dir under checkpoints/ that has a config.json -> it
# needs the final model at ft_head_v1/ ROOT (config.json + model.safetensors + ...).
# Training also writes many 1.26GB per-epoch shards + (was) a doubled checkpoints/ dir.
# If the final root save failed because Drive filled up, rescue the trainer's best_final
# into the root FIRST, then delete the throwaway checkpoints/ subtree.
import shutil, pathlib

ft_head = pathlib.Path(CHECKPOINT_DIR) / "ft_head_v1"
root_missing = not (ft_head / "config.json").exists()

# 1) Free space: delete every per-epoch shard (redundant once best_final exists)
for ckpt in list(ft_head.rglob("checkpoint-*")):
    if ckpt.is_dir():
        mb = sum(f.stat().st_size for f in ckpt.rglob("*") if f.is_file()) / 1e6
        shutil.rmtree(ckpt)
        print(f"[cleanup] deleted {ckpt.name}  freed {mb:.0f} MB")

# 2) Promote the best model to root if the final save had failed (Drive full)
if root_missing:
    bf = sorted([p for p in ft_head.rglob("best_final") if (p / "config.json").exists()],
                key=lambda p: p.stat().st_mtime, reverse=True)
    if bf:
        src = bf[0]
        print(f"[cleanup] rescuing final model from {src}")
        for f in src.iterdir():
            dst = ft_head / f.name
            if f.is_file() and not dst.exists():
                shutil.copy2(f, dst)
    else:
        print("[cleanup] !! no best_final to rescue — model may be unrecoverable")

# 3) Delete the throwaway checkpoints/ subtree (epoch shards, best_final copy, doubled dir)
ck = ft_head / "checkpoints"
if ck.exists():
    mb = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e6
    shutil.rmtree(ck)
    print(f"[cleanup] deleted {ck}  freed {mb:.0f} MB")

print("\n[cleanup] final model at root:", "OK" if (ft_head/"config.json").exists() else "MISSING")
print("[cleanup] ft_head_v1 contents:", sorted(p.name for p in ft_head.iterdir()))


In [ ]:
# @title 5. A/B: base ASVspoof vs fine-tuned, on the SAME unseen speaker
# TWO SEPARATE CLAIMS — do NOT conflate these on the slide:
#   * LOSO            = N-fold CV over ALL speakers (mean +/- std), train-only-the-head.
#   * TRUE HOLDOUT    = 1 speaker kept ENTIRELY out of the final model, single run.
# Below: run the PRETRAINED (no --checkpoint) model on the true-holdout speaker's clips,
# compare with the fine-tuned holdout numbers from finetune_report.json => measured lift.
# Guard: requires vars from cells 1-2. If you disconnected, RUN CELLS 1 AND 2 FIRST
# (cell 2 is cheap - no training, just rebuilds the holdout dir). You can SKIP cell 3
# (training); this cell reads the saved finetune_report.json.
for _req in ("SRC_PKG", "holdout_dir", "results_dir", "CHECKPOINT_DIR"):
    if _req not in dir():
        raise RuntimeError(f"'{_req}' not defined. Run Cell 1 and Cell 2 first "
                           "(they define it and are fast; skip the training Cell 3).")

import subprocess, sys, json, glob


out = str(results_dir / "sprint3_base_asvspoof_holdout.json")
r = subprocess.run([
    sys.executable, "-m", "evaluate",
    "--root", str(holdout_dir), "--variant", "wav2vec2",
    "--out", out, "--find-threshold",
], cwd=str(SRC_PKG))
print("base ASVspoof eval ->", out)

ft = json.loads(pathlib.Path(CHECKPOINT_DIR / "ft_head_v1" / "finetune_report.json").read_text())
base = json.loads(pathlib.Path(out).read_text()) if r.returncode == 0 and pathlib.Path(out).exists() else None
# Normalize base metrics to flat acc/fpr/fnr — works for both payload shapes:
#   mode=find-threshold -> {"best": {acc,fpr}} (older) and mode=evaluate -> flat keys.
if base:
    base = dict(base)
    if "accuracy" not in base and base.get("best"):
        base["accuracy"] = base["best"].get("acc")
        base["fpr"]      = base["best"].get("fpr")
        base["fnr"]      = None  # not emitted at best-level by --find-threshold

print("\n=================== CLAIM 2: TRUE HOLD-OUT A/B (1 unseen speaker)"
      " ===================")
if base and base.get("accuracy") is not None:
    print(f"  model: base ASVspoof : acc={base['accuracy']:.3f} fpr={base['fpr']:.3f} fnr={base.get('fnr')}")
else:
    print("  base ASVspoof: <no usable JSON — check evaluate.py output>")
if isinstance(ft.get("holdout"), dict):
    h = ft["holdout"]
    print(f"  model: fine-tuned    : acc={h['finetuned_eval']['acc']:.3f} fpr={h['finetuned_eval']['fpr']:.3f} fnr={h['finetuned_eval']['fnr']:.3f}")
else:
    print("  fine-tuned    : <no holdout in report — did you pass --holdout?>")

print("\n=================== CLAIM 1: LOSO aggregate (N-fold CV, all speakers)"
      " ===================")
l = ft.get("loso") or {}
if l:
    for k in ("acc", "fpr", "fnr"):
        print(f"    {k}: {l[k]['mean']:.3f} +/- {l[k]['std']:.3f}   per-fold={[round(v,3) for v in l[k]['per_fold']]}")
print("\nNOTE: the holdout speaker above was ALSO one of the LOSO folds. Keep these")
print("two claims separate on the slide. Copy numbers into results.md.")